In [0]:
%sql
WITH revenue_df AS (
  SELECT *
  FROM global_tech_gold.global_tech_fact_tables.fact_finance
  WHERE account_type = 'Revenue'
),
monthly_revenue AS (
  SELECT
    TO_DATE(DATE_TRUNC('month', posting_date)) AS month,
    ROUND(SUM(amount), 2) AS total_revenue
  FROM revenue_df
  GROUP BY TO_DATE(DATE_TRUNC('month', posting_date))
),
monthly_revenue_lag AS (
  SELECT
    month,
    total_revenue,
    ROUND(LAG(total_revenue) OVER (ORDER BY month), 2) AS prev_month_revenue
  FROM monthly_revenue
)
SELECT
  month,
  total_revenue,
  prev_month_revenue,
  ROUND(((total_revenue - prev_month_revenue) / prev_month_revenue) * 100, 2) AS mom_growth_percentage
FROM monthly_revenue_lag

In [0]:
%sql
SELECT
  company_id,
  TO_DATE(DATE_TRUNC('month', posting_date)) AS month,
  ROUND(SUM(amount), 2) AS cost_of_sales
FROM global_tech_gold.global_tech_fact_tables.fact_finance
GROUP BY company_id, TO_DATE(DATE_TRUNC('month', posting_date))


In [0]:
%sql
WITH revenue_df AS (
  SELECT
    TO_DATE(DATE_TRUNC('month', posting_date)) AS month,
    ROUND(SUM(amount), 2) AS revenue
  FROM global_tech_gold.global_tech_fact_tables.fact_finance
  WHERE account_type = 'Revenue'
  GROUP BY TO_DATE(DATE_TRUNC('month', posting_date))
),
cogs_df AS (
  SELECT
    TO_DATE(DATE_TRUNC('month', posting_date)) AS month,
    ROUND(SUM(amount), 2) AS cost_of_sales
  FROM global_tech_gold.global_tech_fact_tables.fact_finance
  WHERE account_type LIKE '%COGS%'
  GROUP BY TO_DATE(DATE_TRUNC('month', posting_date))
)
SELECT
  r.month,
  r.revenue,
  c.cost_of_sales,
  CONCAT(ROUND(((c.cost_of_sales - r.revenue) / r.revenue) * 100, 2), '%') AS gross_profit_margin
FROM revenue_df r
JOIN cogs_df c ON r.month = c.month

In [0]:
%sql
SELECT
  company_id,
  account_sk,
  ROUND(SUM(amount), 2) AS total_expense
FROM global_tech_gold.global_tech_fact_tables.fact_finance
WHERE category = 'Operating Expense'
GROUP BY company_id, account_sk

In [0]:
%sql
SELECT
  f.company_id,
  e.position,
  ROUND(AVG(f.total_compensation), 2) AS avg_compensation
FROM global_tech_gold.global_tech_fact_tables.fact_payroll f
JOIN global_tech_gold.global_tech_dim_tables.dim_employee e
  ON f.employee_id = e.employee_id
GROUP BY f.company_id, e.position

In [0]:
%sql
SELECT
  TO_DATE(DATE_TRUNC('month', posting_date)) AS month,
  ROUND(SUM(amount), 2) AS net_profit
FROM global_tech_gold.global_tech_fact_tables.fact_finance
GROUP BY TO_DATE(DATE_TRUNC('month', posting_date))

In [0]:
%sql
SELECT
  d.department_name,
  ROUND(SUM(f.overtime_pay), 2) AS total_overtime,
  ROUND(SUM(f.bonus), 2) AS total_bonus,
  ROUND(SUM(f.overtime_pay) / SUM(f.gross_salary), 4) AS overtime_ratio
FROM global_tech_gold.global_tech_fact_tables.fact_payroll f
JOIN global_tech_gold.global_tech_dim_tables.dim_department d
  ON f.department_id = d.department_id
GROUP BY d.department_name


In [0]:
%sql
SELECT
  d.department_name,
  ROUND(SUM(f.total_compensation), 2) AS department_cost
FROM global_tech_gold.global_tech_fact_tables.fact_payroll f
JOIN global_tech_gold.global_tech_dim_tables.dim_department d
  ON f.department_id = d.department_id
GROUP BY d.department_name

In [0]:
%sql
SELECT
  d.department_name,
  COUNT(*) AS headcount
FROM global_tech_gold.global_tech_dim_tables.dim_employee e
JOIN global_tech_gold.global_tech_dim_tables.dim_department d
  ON e.department_id = d.department_id
WHERE e.termination_date IS NULL
GROUP BY d.department_name

In [0]:
%sql
WITH payroll_cost AS (
  SELECT
    company_id,
    SUM(total_compensation) AS total_payroll_cost
  FROM global_tech_gold.global_tech_fact_tables.fact_payroll
  GROUP BY company_id
),
revenue AS (
  SELECT
    company_id,
    SUM(amount) AS total_revenue
  FROM global_tech_gold.global_tech_fact_tables.fact_finance
  WHERE account_type = 'Revenue'
  GROUP BY company_id
)
SELECT
  p.company_id,
  p.total_payroll_cost,
  r.total_revenue,
  ROUND((p.total_payroll_cost / r.total_revenue) * 100, 2) AS payroll_percentage
FROM payroll_cost p
JOIN revenue r ON p.company_id = r.company_id